## GetONCRawData – Incremental BPR Parquet Pipeline

Fetches raw hex data from the ONC API **one day at a time** (2022-05-23 → today),
parses each ASCII hex line into frequency counts and periods, and appends everything
to a partitioned Parquet dataset at `out/NCHR_BPR_raw.parquet/`.

Rerun at any time to top-up the dataset with the latest days — already-stored days are skipped automatically.

> **Kernel:** select the `bpr-nchr` conda environment.  
> **Auth:** set the `ONC_TOKEN` environment variable (see https://oceannetworkscanada.github.io/api-python-client/ )

In [ ]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()   # loads .env from the project root (safe no-op if file absent)

logging.basicConfig(level=logging.INFO, format='%(levelname)s %(name)s: %(message)s')

token = os.getenv('ONC_TOKEN')
if not token:
    raise EnvironmentError(
        'ONC_TOKEN is not set. Copy .env.example → .env and add your token, '
        'or set the variable in your shell before launching Jupyter.'
    )

In [ ]:
from onc import ONC

onc = ONC(token)

In [ ]:
from src.build_parquet import run_incremental_build, load_dataset

In [ ]:
# ── Configuration ────────────────────────────────────────────────────────────
LOCATION_CODE        = 'NCHR'
DEVICE_CATEGORY_CODE = 'BPR'
START_DATE           = '2022-05-23'  # earliest date to fetch if dataset is empty
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
from datetime import datetime, timezone

run_incremental_build(
    onc_client=onc,
    start_date=datetime.fromisoformat(START_DATE).replace(tzinfo=timezone.utc),
    location_code=LOCATION_CODE,
    device_category_code=DEVICE_CATEGORY_CODE,
)

In [ ]:
# Preview the full dataset
df = load_dataset()
print(f'Total rows : {len(df):,}')
print(f'Date range : {df.index.min()}  →  {df.index.max()}')
df.head()

In [ ]:
# Optional: load a specific date range
# df_sub = load_dataset(date_from='2025-01-01', date_to='2025-02-01')
# df_sub.head()